In [7]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Install FastAPI and Uvicorn for the bridge
!pip install fastapi uvicorn pyngrok python-multipart

# Start Ollama in the background
import subprocess
import time
ollama_process = subprocess.Popen(["ollama", "serve"])
time.sleep(5) # Wait for startup

# Pull a model optimized for Linux/Coding tasks
!ollama pull llama3:8b

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [6]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (604 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 117540 files and directories currently in

In [9]:
!ollama list

NAME         ID              SIZE      MODIFIED       
llama3:8b    365c0bd3c000    4.7 GB    13 seconds ago    


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import requests

app = FastAPI()

class Task(BaseModel):
    prompt: str

@app.post("/generate_command")
def get_command(task: Task):
    # System Prompt: Tells the AI to ONLY output the bash command
    system_prompt = "You are a Linux System Expert. Return ONLY the bash command to solve the user's request. No explanations."
    
    response = requests.post("http://localhost:11434/api/generate", json={
        "model": "llama3:8b",
        "prompt": f"{system_prompt}\nUser Request: {task.prompt}",
        "stream": False
    })
    
    return {"command": response.json()['response'].strip()}

# Run the server in a separate thread
import threading
import uvicorn

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_api).start()

INFO:     Started server process [341]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [13]:
# Create a public tunnel for our FastAPI on port 8000
!ssh -o StrictHostKeyChecking=no -p 443 -R0:localhost:8000 qr@a.pinggy.io

Allocated port 3 for remote forward to localhost:8000
7=)0]8;;\                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         ┌────────────────────────────┐                                                  │                            │                                                  │ Wait while we prepare the  │                                                  │             UI           